1 Load the Dataset

In [ ]:
!pip install dataset

In [ ]:
!pip install -U dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

2 Tokenize the dataset 

2.1 Tokenize the dataset into tokenIDs
2.2 Create a file called "train.bin" and "validation.bin" where we will store the tokenIDs for the entire dataset.
2.3 We make sure the tokenIDs are stored on a disk, rather than on RAM for efficient computation.

In [ ]:
!pip install tiktoken
import tiktoken
import os 
import numpy as np

from tqdm.auto import tqdm

enc = tiktoken.get_encoding("gpt2")

#  Some functions from https://github.com/karpathy/nanoGPT/blob/master/data/openwebtext/prepare.py

def process(example):
    # encode_ordinary ignores any special token
    ids = enc.encode_ordinary(example["text"])
    out = {{"ids":ids, "len": len(ids)}}

    return out

if not os.path.exists("train.bin"):
    tokenized = ds.map(
        process,
        remove_columns=["text"],
        desc="tokeninzing the splits",
        num_proc=8,
    )

    # concatenate all the ids in each dataset into one large file we use for training
    for split, dset in tokenized.items():
        arr_len = np.sum(dset["len"], dtype=np.uint64)
        filename = f"{split}.bin"
        dtype = np.uint16 # can do since enc.max_token_value == 50256 is < 2**16
        arr = np.memmap(filename, dtype=dtype, mode="w+", shape=(arr_len,))
        total_batches = 1024

        idx = 0
        for batch_idx in tqdm(range(total_batches), desc=f"writing {filename}"):
            # batch together samples for faster write
            batch = dset.shard(num_shard=total_batches, index=batch_idx, contiguous=True)
            arr_batch = np.concatenate(batch["ids"])
            # write into map 
            arr[idx: idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)

        arr.flush()